# Enkidu 실험 환경 (Colab)

목표: 화자별 UFP 패치를 학습해 보호본을 만들고, 보호가 실제로 걸렸는지 먼저 확인한다.

이 노트북이 하는 일
1. 의존성 설치와 Enkidu 저장소 클론
2. 데이터 점검 (화자별 조각 수가 충분한지)
3. 화자별 UFP 학습 (학습 70% / 평가 30% 분리)
4. 평가용 조각에 패치를 붙여 보호본 생성
5. **검증: 원본 vs 보호본 ECAPA 유사도** — 여기서 유사도가 떨어지지 않으면 뒤 실험은 의미가 없다

런타임은 반드시 GPU로 설정한다. 런타임 > 런타임 유형 변경 > T4 GPU

## 1. 환경 확인과 설치

Enkidu README는 torch 2.4.0 고정을 권하지만, 코드가 쓰는 API는 표준 범위다.
Colab 기본 torch를 그대로 쓰고 speechbrain만 설치한다. torch를 바꾸면 설치에 수 분이 걸리고 CUDA 조합이 깨질 수 있다.

In [ ]:
import torch, sys
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "런타임 유형을 GPU로 바꿔 주세요"

In [ ]:
# speechbrain 1.0 이상이어야 speechbrain.inference 경로가 있다
!pip -q install "speechbrain>=1.0.0" pystoi
import speechbrain
print("speechbrain :", speechbrain.__version__)

## 2. 구글 드라이브 마운트와 경로 설정

팀 공용 폴더 구조를 쓴다. D1~D3 모두 화자별 하위 폴더를 두고, 파일명은 D1과 똑같이 맞춘다.

```
MyDrive/xtts_exp/
├── D1/<화자>/...             원본      ┐
├── D2/<화자>/...             보호본    ├ XTTS 입력
├── D3/<화자>/...             정화본    ┘
├── ufp/                      보호 산출물
│   ├── <화자>.pt             화자별 패치
│   ├── config.json           설정 기록
│   └── protection_check.csv  D1 vs D2 검증 결과
└── runs/run1/clones/         XTTS 결과
```

D1~D3는 공용 데이터, `runs/<RUN_ID>/`는 그 회차의 산출물이다.
패치를 회차 안에 두면 어떤 패치로 만든 복제본인지 나중에 역추적할 수 있다.
설정을 바꿔 다시 돌릴 때는 `RUN_ID`만 바꾸면 이전 회차가 보존된다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

RUN_ID = 'run1'        # 설정을 바꿔 다시 돌릴 때는 여기만 바꾼다

ROOT      = Path('/content/drive/MyDrive/xtts_exp')
SEG_DIR   = ROOT / 'D1'                           # 원본 (화자별 폴더)
PROT_DIR  = ROOT / 'D2'                           # 보호본 출력 (같은 구조)
RUN_DIR   = ROOT / 'runs' / RUN_ID                # 이 회차의 산출물
UFP_DIR   = RUN_DIR / 'enkidu_ufp'                # 학습한 패치 저장

for d in (PROT_DIR, UFP_DIR, RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert SEG_DIR.exists(), f"D1 폴더가 없습니다: {SEG_DIR}"
print("D1 (원본)  :", SEG_DIR)
print("D2 (보호본):", PROT_DIR)
print("runs       :", RUN_DIR)

## 3. Enkidu 클론과 패치

공개 코드에 그냥 쓰면 걸리는 점이 몇 개 있다.

| 문제 | 실제 | 이 노트북의 처리 |
| --- | --- | --- |
| README는 `cli_enkidu.py` | 실제 파일명은 `enkidu_cli.py` | CLI 대신 Python API를 쓴다 |
| CLI `--noise_level` 기본값 0.1 | 논문 기본값은 0.4 | 명시적으로 넘긴다 |
| `forward()`가 샘플마다 print | 70샘플 × 50에폭 = 3500줄 | stdout을 막고 직접 로그를 찍는다 |
| `__init__`의 `self.augmentation = augmentation` | 같은 이름의 메서드를 덮어쓴다 | `forward()`가 그 메서드를 호출하지 않아 무해. 단 Algorithm 1의 TemporalAugmentation은 실제로 실행되지 않는다 |

**코드를 직접 읽고 추가로 확인한 것** (`core/enkidu.py`)

- 학습 중 `add_noise`는 `random_offset=False`로 호출된다. 실제로 켜진 학습 증강은 프레임 마스킹(`mask_ratio`) 하나뿐이다
- `hop_length` 기본값은 512라 패치 30프레임은 약 0.96초다
- `noise_smooth=True`의 Wiener 필터와 손실의 지각 항(λ=0.1)은 결과에 거의 영향이 없었다 (`variants/v1_masking` 진단 실험)

In [ ]:
%cd /content
!rm -rf Enkidu
!git clone -q https://github.com/NoobCodeNameless/Enkidu.git
%cd /content/Enkidu
!ls

In [ ]:
import sys
sys.path.insert(0, '/content/Enkidu')

from core import Enkidu
from wavdataset import WaveformPrivacyDataset
print("import OK")

## 4. 데이터 점검

논문 절제 실험(Figure 5)에서 학습 데이터가 10%면 ECAPA SPR이 36%까지 떨어지고, 70%에서 100%에 수렴한다.
논문은 화자 1명당 발화 100개(평균 4.94초, 약 8분)를 썼고 그중 70개로 학습했다.

**화자당 조각이 40개 미만이면 패치가 제대로 학습되지 않을 수 있다.** 이 셀에서 먼저 확인한다.

In [ ]:
import torchaudio
import soundfile as sf

SR = 16000

ds = WaveformPrivacyDataset(
    dataset_dir=str(SEG_DIR),
    sample_rate=SR,
    mono=True,
    wav_format='wav',     # 기본 예시는 LibriSpeech라 flac이다
)

print(f"화자 {len(ds.speakers)}명, 전체 조각 {len(ds)}개\n")
print(f"{'화자':<14}{'조각':>6}{'총 길이':>10}{'평균':>8}   판정")
print("-" * 54)

for spk in ds.speakers:
    paths = ds.speaker_samples[spk]
    total = 0.0
    for p in paths:
        total += sf.info(p).duration      # torchaudio.info는 새 버전에서 제거됨
    n = len(paths)
    avg = total / n if n else 0
    verdict = "충분" if n >= 40 else ("부족 - 추가 녹음 권장" if n < 20 else "빈약")
    print(f"{spk:<14}{n:>6}{total/60:>9.1f}분{avg:>7.1f}초   {verdict}")

## 5. UFP 학습

화자마다 한 번만 하면 된다. 패치는 `results/enkidu_ufp/<이름>.pt`에 저장되므로 다시 돌릴 필요가 없다.

설정은 논문 기본값을 따른다.

| 인자 | 값 | 비고 |
| --- | --- | --- |
| `noise_level` | 0.4 | 논문 Table 4에서 보호와 품질의 균형점 |
| `frame_length` | 30 | 약 1초. 논문 Figure 4에서 30과 300이 좋고 60~120이 나쁘다 |
| `mask_ratio` | 0.3 | 학습 중 패치를 건너뛰는 비율 |
| `steps` | 50 | 논문 본문은 50~100 수렴. 100은 0.2%p 개선에 시간 두 배 |
| `alpha` | 0.1 | Adam 학습률 |
| `TRAIN_RATIO` | 0.7 | 학습 70% / 평가 30% |

In [ ]:
NOISE_LEVEL  = 0.4
FRAME_LENGTH = 30
MASK_RATIO   = 0.3
STEPS        = 50
ALPHA        = 0.1
TRAIN_RATIO  = 0.7
DEVICE       = 'cuda:0'

def split_speaker(spk):
    """화자의 조각을 학습용 / 평가용으로 겹치지 않게 나눈다."""
    paths = sorted(ds.speaker_samples[spk])
    k = max(1, int(len(paths) * TRAIN_RATIO))
    return paths[:k], paths[k:]

for spk in ds.speakers:
    tr, ev = split_speaker(spk)
    print(f"{spk:<14} 학습 {len(tr):>3}개   평가 {len(ev):>3}개")

In [ ]:
import io, time, contextlib
from speechbrain.inference import SpeakerRecognition
import torch.nn.functional as F

# ECAPA 하나를 대리 모델과 검증에 같이 쓴다.
# Enkidu가 공략하는 모델이므로 여기 수치는 낙관적으로 나온다는 점을 기억할 것.
ecapa = SpeakerRecognition.from_hparams(
    'speechbrain/spkrec-ecapa-voxceleb',
    savedir='/content/pretrained/ecapa',
    run_opts={"device": DEVICE},
)

def load_wav(path):
    w, sr = torchaudio.load(path)
    if w.shape[0] > 1:
        w = w.mean(dim=0, keepdim=True)
    if sr != SR:
        w = torchaudio.transforms.Resample(sr, SR)(w)
    return w

@torch.no_grad()
def ecapa_sim(a, b):
    ea = ecapa.encode_batch(a.reshape(1, -1).to(DEVICE)).reshape(1, -1)
    eb = ecapa.encode_batch(b.reshape(1, -1).to(DEVICE)).reshape(1, -1)
    return F.cosine_similarity(ea, eb).item()

In [ ]:
# 기울기가 실제로 패치까지 흐르는지 먼저 확인한다 (30초)
_probe = Enkidu(model=ecapa, steps=1, alpha=ALPHA, mask_ratio=MASK_RATIO,
                frame_length=FRAME_LENGTH, noise_level=NOISE_LEVEL, device=DEVICE)
_nr = torch.randn((1, 513, FRAME_LENGTH), requires_grad=True, device=DEVICE)
_ni = torch.randn((1, 513, FRAME_LENGTH), requires_grad=True, device=DEVICE)
_x  = load_wav(ds.speaker_samples[ds.speakers[0]][0])
_y  = _probe.add_noise(_x, _nr, _ni, mask_ratio=0.0, random_offset=False, noise_smooth=True)
_probe.total_loss(_x.to(DEVICE), _y).backward()
print("grad 흐름:", _nr.grad is not None and _nr.grad.abs().sum().item() > 0)
del _probe, _nr, _ni, _x, _y
torch.cuda.empty_cache()

In [ ]:
ufps = {}

for spk in ds.speakers:
    pt = UFP_DIR / f'{spk}.pt'
    if pt.exists():
        d = torch.load(pt)
        ufps[spk] = (d['real'], d['imag'])
        print(f"[{spk}] 기존 패치 사용 ({pt.name})")
        continue

    train_paths, _ = split_speaker(spk)
    samples = [load_wav(p) for p in train_paths]

    model = Enkidu(
        model=ecapa,
        steps=STEPS,
        alpha=ALPHA,
        mask_ratio=MASK_RATIO,
        frame_length=FRAME_LENGTH,
        noise_level=NOISE_LEVEL,
        noise_smooth=True,
        device=DEVICE,
    )

    t0 = time.time()
    with contextlib.redirect_stdout(io.StringIO()):   # 샘플마다 찍는 로그를 막는다
        nr, ni = model(samples)
    dt = time.time() - t0

    torch.save({'real': nr, 'imag': ni,
                'noise_level': NOISE_LEVEL, 'frame_length': FRAME_LENGTH,
                'mask_ratio': MASK_RATIO, 'steps': STEPS,
                'n_train': len(samples)}, pt)
    ufps[spk] = (nr, ni)
    print(f"[{spk}] 학습 {len(samples)}개 · {dt/60:.1f}분 · 저장 {pt.name}")

## 6. 보호본 생성

배포 모드다. 학습 때와 달리 마스킹과 시간 이동을 끄고 전역에 붙인다(`mask_ratio=0.0`, `random_offset=False`).

두 가지를 지킨다.

- **평가용 조각에만** 적용한다. 패치 학습에 쓴 조각으로 평가하면 수치가 부풀려진다
- **파일명과 화자 폴더를 D1과 똑같이** 유지한다. D1/D2/D3를 파일명으로 짝지어야 조건별 비교가 된다

In [ ]:
applier = Enkidu(model=ecapa, steps=1, alpha=ALPHA, mask_ratio=MASK_RATIO,
                 frame_length=FRAME_LENGTH, noise_level=NOISE_LEVEL,
                 noise_smooth=True, device=DEVICE)

made = 0
for spk in ds.speakers:
    nr, ni = ufps[spk]
    _, eval_paths = split_speaker(spk)
    out_dir = PROT_DIR / spk
    out_dir.mkdir(parents=True, exist_ok=True)

    for p in eval_paths:
        x = load_wav(p)
        y = applier.add_noise(x, nr, ni,
                              mask_ratio=0.0,        # 배포: 전역 타일링
                              random_offset=False,
                              noise_smooth=True)
        torchaudio.save(str(out_dir / Path(p).name), y.cpu(), SR)
        made += 1

    print(f"[{spk}] 보호본 {len(eval_paths)}개 -> {out_dir}")

print(f"\n합계 {made}개")

## 7. 검증 — 보호가 실제로 걸렸는가

가장 중요한 셀이다. 복제를 돌리기 전에 **원본 vs 보호본**의 ECAPA 유사도를 본다.

- 논문대로면 유사도가 뚜렷하게 떨어져야 한다
- 거의 안 떨어지면 데이터가 부족하거나 설정이 잘못된 것이다. `noise_level`을 올리거나 조각을 더 모은다
- 참고 임계값: De-AntiFake가 쓴 SpeechBrain 기본값 0.25

STOI도 같이 본다. 논문은 0.71을 보고했는데 이건 높은 값이 아니다.

In [ ]:
import numpy as np
from pystoi import stoi
import pandas as pd

rows = []
for spk in ds.speakers:
    _, eval_paths = split_speaker(spk)
    for p in eval_paths:
        name = Path(p).name
        q = PROT_DIR / spk / name
        if not q.exists():
            continue
        x = load_wav(p)
        y = load_wav(str(q))
        n = min(x.shape[-1], y.shape[-1])
        rows.append({
            'speaker': spk,
            'file': name,
            'sim': ecapa_sim(x, y),
            'stoi': stoi(x[0, :n].numpy(), y[0, :n].numpy(), SR, extended=False),
        })

df = pd.DataFrame(rows)
summary = df.groupby('speaker').agg(
    n=('sim', 'size'), sim_mean=('sim', 'mean'), sim_std=('sim', 'std'),
    stoi_mean=('stoi', 'mean')).round(3)
print(summary.to_string())
print(f"\n전체 평균  유사도 {df['sim'].mean():.3f}   STOI {df['stoi'].mean():.3f}")
print(f"유사도 범위 {df['sim'].min():.3f} ~ {df['sim'].max():.3f}")

df.to_csv(RUN_DIR / 'enkidu_protection_check.csv', index=False)
print(f"저장: {RUN_DIR / 'enkidu_protection_check.csv'}")

In [ ]:
# 판정
m = df['sim'].mean()
if m > 0.9:
    print(f"유사도 {m:.3f} — 보호가 거의 안 걸렸습니다.")
    print("  noise_level을 0.5~0.7로 올리거나, 화자당 조각을 40개 이상으로 늘려 다시 학습하세요.")
elif m > 0.7:
    print(f"유사도 {m:.3f} — 약하게 걸렸습니다. noise_level 상향을 검토하세요.")
else:
    print(f"유사도 {m:.3f} — 임베딩 수준에서 보호가 걸렸습니다. 복제 실험으로 넘어가세요.")
print(f"STOI {df['stoi'].mean():.3f} (논문 보고값 0.71, 0.85 이상이면 명료도 양호)")

## 8. 들어보기

보호본이 실제로 어떻게 들리는지 확인한다. 사람 귀에 거의 안 들린다는 게 논문 주장이다.

In [ ]:
from IPython.display import Audio, display

spk = ds.speakers[0]
_, ev = split_speaker(spk)
name = Path(ev[0]).name
print(f"{spk} / {name}")
print("원본"); display(Audio(ev[0]))
print("보호본"); display(Audio(str(PROT_DIR / spk / name)))

## 다음 단계

이 노트북은 5개 조건 중 **4번(보호본 자체 점검)**까지만 한다. 남은 것은 다음과 같다.

| # | 조건 | 입력 | 상태 |
| --- | --- | --- | --- |
| 1 | 원본 → XTTS | D1 | 복제 노트북 |
| 2 | 보호본 → XTTS | D2 | 복제 노트북 |
| 3 | 보호본 → De-AntiFake → XTTS | D3 | 정화 노트북 후 복제 |
| 4 | 보호본 자체 (복제 없이) | D1 vs D2 | **이 노트북** |
| 5 | 원본 → De-AntiFake → XTTS | 대조군 | 정화 노트북 후 복제 |

5번은 D3 규칙에 안 들어가는 대조군이다. 정화 자체가 음성을 망가뜨리지 않는다는 걸 보여야 3번 결과를 "정화가 보호만 지웠다"고 말할 수 있으므로, `runs/` 아래 따로 두는 쪽을 권한다.

주의할 점

- 판정 임계값 τ는 **모든 조건에 같은 값**을 쓴다. 화자 3명으로 구한 0.798은 EER이 0%라 극단값이니, 화자를 늘려 다시 구하는 쪽을 권한다
- 등록용 음성과 XTTS 참조로 준 조각이 겹치면 점수가 부풀려진다
- XTTS는 돌릴 때마다 결과가 달라지므로 시드를 여러 개 써서 평균을 낸다
- De-AntiFake는 한국어에서 정화 단계만 쓴다 (`--purification_methods DDPM`). 다듬기 단계는 영어 대본과 MFA가 필요하다

기억할 것: Enkidu도 XTTS를 평가하지 않았고 대리 모델이 ECAPA 하나뿐이다. 2번에서 복제가 성공해도 실패가 아니라 **기존 제로샷 방어가 XTTS에 약하다**는 근거가 된다.